# Week 5 — 迴歸與因子模型

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 用矩陣形式推導並實作 OLS。
- 估計 CAPM 式市場 beta 並解讀。
- 配適多因子模型並檢視殘差。
- 計算 rolling beta 並理解 omitted variable bias。

## 預估學習時間

約 9–11 小時。

## 先備概念

- Week 1 的矩陣運算
- Week 4 的標準誤

## 外部學習資源

- [NTU OpenCourseWare 統計學一上與計量導論](https://ocw.aca.ntu.edu.tw/courses/112S103)
- [MIT OpenCourseWare 18.06SC Linear Algebra](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

In [ ]:
# 教學樣式設定（CJK 字型、負號正常顯示、固定隨機種子）
import matplotlib as _mpl
_mpl.rcParams['font.sans-serif'] = [
    'PingFang TC', 'Heiti TC', 'Microsoft JhengHei',
    'Noto Sans CJK TC', 'Noto Sans TC',
    'WenQuanYi Zen Hei', 'Source Han Sans TC',
    'Arial Unicode MS', 'DejaVu Sans',
]
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## 概念說明

### OLS 的矩陣形式

模型 $y = X\beta + \varepsilon$ 的最小平方解為

$$ \hat\beta = (X^\top X)^{-1} X^\top y. $$

幾何上，$X\hat\beta$ 是把 $y$ **投影**到 $X$ 各欄所張成的子空間；殘差 $y - X\hat\beta$ 與該子空間正交。

### 財務意義

CAPM 式迴歸 $r_{\text{asset}} = \alpha + \beta\, r_{\text{market}} + \varepsilon$ 中，$\beta$ 衡量資產對市場的曝險。**但要切記：迴歸係數不會自動變成可交易的訊號**——它只是描述歷史共變關係。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from quant_math_roadmap.math.statistics import ols_fit
from quant_math_roadmap.math.linear_algebra import add_intercept, ols_beta

rng = np.random.default_rng(2024)
n = 600
market = rng.normal(0.0003, 0.011, n)
true_beta, true_alpha = 1.2, 0.0001
idiosyncratic = rng.normal(0.0, 0.008, n)
asset = true_alpha + true_beta * market + idiosyncratic
print('已產生合成市場與資產報酬，n =', n)

### 手刻 OLS vs statsmodels

In [ ]:
fit = ols_fit(market, asset, add_const=True, feature_names=['market'])
print(fit.summary())
print()
sm_fit = sm.OLS(asset, sm.add_constant(market)).fit()
print('statsmodels 係數:', np.round(sm_fit.params, 6))
print('我們的係數    :', np.round(fit.params, 6))
assert np.allclose(fit.params, sm_fit.params)

估計的 beta 應接近真實值 1.2。手刻 OLS 與 `statsmodels` 完全吻合，印證 $\hat\beta = (X^\top X)^{-1}X^\top y$。

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(market, asset, s=8, alpha=0.4, label='觀測值')
grid = np.linspace(market.min(), market.max(), 100)
ax.plot(grid, fit.params[0] + fit.params[1] * grid,
        label=f'OLS 配適線 (beta={fit.params[1]:.3f})')
ax.set_title('CAPM 式迴歸：資產報酬 vs 市場報酬')
ax.set_xlabel('市場報酬')
ax.set_ylabel('資產報酬')
ax.legend()
plt.show()

### 多因子模型

In [ ]:
value_factor = rng.normal(0.0, 0.007, n)
size_factor = rng.normal(0.0, 0.006, n)
asset_multi = (0.0001 + 1.1 * market + 0.6 * value_factor
               - 0.3 * size_factor + rng.normal(0, 0.005, n))
X = np.column_stack([market, value_factor, size_factor])
multi_fit = ols_fit(X, asset_multi, add_const=True,
                    feature_names=['market', 'value', 'size'])
print(multi_fit.summary())

每個係數是「在其他因子固定下」該因子的曝險。$R^2$ 衡量模型解釋了多少報酬變異——但高 $R^2$ **不**代表可獲利。

### Rolling beta

In [ ]:
asset_s = pd.Series(asset)
market_s = pd.Series(market)
window = 120
rolling_beta = []
for end in range(window, n + 1):
    sl = slice(end - window, end)
    b = ols_beta(add_intercept(market_s.iloc[sl].to_numpy()),
                 asset_s.iloc[sl].to_numpy())
    rolling_beta.append(b[1])
rolling_beta = pd.Series(rolling_beta, index=range(window, n + 1))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(rolling_beta.index, rolling_beta.values, label=f'{window} 期 rolling beta')
ax.axhline(true_beta, linestyle='--', label=f'真實 beta = {true_beta}')
ax.set_title('Rolling beta 隨時間的估計')
ax.set_xlabel('視窗結束位置')
ax.set_ylabel('估計 beta')
ax.legend()
plt.show()

即使真實 beta 固定，rolling 估計仍會在其周圍震盪——這就是估計的抽樣不確定性。真實資料的 beta 還會**真的隨時間改變**。

### Heteroskedasticity 與穩健標準誤（HC0 / HC1）

金融資料常見**異質變異**：誤差的變異數不是常數（例如隨市場波動放大）。此時 OLS 的**係數估計仍然不偏**，但古典標準誤失準——顯著性檢定會被誤導。White (1980) 的 **sandwich 估計**只用「實際殘差的平方」重新估計係數的共變異數，不需要假設誤差結構：

$$ \widehat{\mathrm{Var}}(\hat\beta)_{HC0} = (X^\top X)^{-1} X^\top \mathrm{diag}(e_i^2)\, X (X^\top X)^{-1} $$

下面刻意製造一組誤差變異隨 $|x|$ 放大的資料，比較古典與穩健標準誤。

In [ ]:
# 誤差標準差 = 0.5 + |x| -> 教科書級的 heteroskedasticity
x_het = rng.standard_normal(800)
y_het = 1.0 + 2.0 * x_het + rng.standard_normal(800) * (0.5 + np.abs(x_het))

classic = ols_fit(x_het, y_het, feature_names=['x'])
robust = ols_fit(x_het, y_het, feature_names=['x'], robust='HC1')
print('係數完全相同:', np.allclose(classic.params, robust.params))
print(f'斜率的古典標準誤   = {classic.std_errors[1]:.4f}')
print(f'斜率的 HC1 穩健標準誤 = {robust.std_errors[1]:.4f}')
print('異質變異下，古典標準誤明顯低估不確定性 -> t 統計量被高估。')

**結論**：報告金融迴歸時，預設使用穩健標準誤是良好習慣。注意它只修推論（標準誤、t、p-value），不改變係數本身——模型的解釋力沒有變，變的是你對顯著性的信心。

### 殘差檢視與 omitted variable bias

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(range(len(multi_fit.residuals)), multi_fit.residuals, s=8, alpha=0.4)
ax.axhline(0.0, linestyle='--')
ax.set_title('多因子模型的殘差')
ax.set_xlabel('觀測索引')
ax.set_ylabel('殘差')
plt.show()

In [ ]:
# 故意遺漏 value 因子，看 beta 如何被扭曲
biased = ols_fit(market, asset_multi, add_const=True, feature_names=['market'])
full = ols_fit(X, asset_multi, add_const=True,
               feature_names=['market', 'value', 'size'])
print('遺漏變數時 market 係數:', round(biased.params[1], 4))
print('完整模型   market 係數:', round(full.params[1], 4))
print('若遺漏變數與納入變數相關，估計係數就會有偏誤。')

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用幾何投影的語言解釋 OLS 在做什麼。
2. 解釋截距、beta、殘差、$R^2$ 各自代表什麼。
3. 為什麼「迴歸係數顯著」不等於「可以拿來交易」？

### 應用練習

In [ ]:
# 應用練習 1：不要用 ols_fit，直接用矩陣公式 (X^T X)^-1 X^T y 估計 beta，
# 並與 ols_fit 比對（記得加截距欄）。
Xc = add_intercept(market)
my_beta = None  # TODO: np.linalg.solve(Xc.T @ Xc, Xc.T @ asset)
if my_beta is not None:
    print('手算 beta:', np.round(my_beta, 6))

In [ ]:
# 應用練習 2：把 rolling 視窗改成 60 期，觀察 rolling beta 的波動如何變化。
win = 60
betas = []  # TODO: 仿照上面用 win 計算 rolling beta
print('完成後比較 60 期與 120 期的波動度。')

### 反思問題

1. 假設你用迴歸發現某因子對下一期報酬「顯著」。在把它變成回測訊號之前，Week 4（多重檢定）與 Week 8（leakage）各提醒你要注意什麼？

## 小測驗（自我檢核）
回答下面的選擇題，然後執行下一格自動對答案。答案以雜湊儲存，不會直接洩漏。

**Q1. OLS 的矩陣解 β̂ 是？**
- A. (XᵀX)⁻¹Xᵀy
- B. Xᵀy
- C. X⁻¹y
- D. (XXᵀ)⁻¹yX

**Q2. R² 衡量的是？**
- A. 策略可獲利程度
- B. 模型解釋的應變數變異比例
- C. 係數的大小
- D. 殘差的總和

**Q3. heteroskedasticity（異質變異）主要影響 OLS 的？**
- A. 係數估計值
- B. 標準誤（以及 t 統計量）
- C. R²
- D. 截距

**Q4. HC0/HC1 穩健標準誤改變的是？**
- A. 迴歸係數
- B. 標準誤
- C. 殘差
- D. R²

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: 填入 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '881e66b9d3c17d9a', 2: '07f7b302f3c9c15d', 3: '2e30a5ff5a744c2f', 4: 'b22e68d5e3df82b5'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: 未作答')
        continue
    _h = _hashlib.sha256(f'qmr-w5-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ 正確' if _ok else '✘ 不正確'))
print(f'得分: {_n_correct} / {len(my_answers)}')

## 常見錯誤

- **忘記在設計矩陣加入截距欄。**
- **把高 $R^2$ 當成策略可獲利的證據。**
- **忽略 heteroskedasticity 使一般 OLS 標準誤失準。**
- **遺漏重要變數造成 omitted variable bias。**
- **把迴歸係數直接當成可交易訊號。**

## 完成本週後，你應該能做到什麼

- [ ] 能推導並實作 $\hat\beta=(X^\top X)^{-1}X^\top y$。
- [ ] 能解讀截距、beta、殘差與 $R^2$。
- [ ] 能計算 rolling beta 並解釋其波動。
- [ ] 能說明 omitted variable bias。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。